In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# imports for time series
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:
df = pd.read_csv('/Users/Marcy_Student/Desktop/marcy/cleaned data.csv')
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", 100)
df.head()
df.drop(['Unnamed: 0'],axis=1, inplace=True) # drop additionaal non needed column
df.keys()
df['mod_zcta']=df['mod_zcta'].astype(str)
df['date']=pd.to_datetime(df['date'])
df.dtypes # check data types

extract_date                  object
date                  datetime64[ns]
mod_zcta                      object
total_ed_visits                int64
ili_pne_visits                 int64
ili_pne_admissions             int64
dtype: object

## 1. Model Pipeline for zipcode

In [21]:
def fit_pipeline(df, zip_code):
    """Run the full forecast pipeline for one zipcode (monthly)."""
    # Filter & aggregate monthly
    ts = df[df['mod_zcta'] == zip_code].groupby('date')['ili_pne_visits'].sum()
    ts = ts.resample('MS').sum()

    # Skip if series is too short or constant
    if len(ts) < 6 or ts.nunique() <= 1:
        return None

    # --- Evaluation: train/test split (80/20) ---
    split = int(len(ts) * 0.8)
    train, test = ts[:split], ts[split:]

    # Skip if train or test is constant
    if train.nunique() <= 1 or test.nunique() <= 1:
        return None

    # Naive baseline
    naive_forecast = pd.Series(train.iloc[-1], index=test.index)

    # Check stationarity with ADF test to determine d
    adf_pvalue = adfuller(train)[1]
    d = 0 if adf_pvalue < 0.05 else 1

    # ARIMA on train split for evaluation
    eval_model = ARIMA(train, order=(1, d, 1))
    eval_fit = eval_model.fit()
    eval_forecast = eval_fit.forecast(steps=len(test))

    naive_rmse = np.sqrt(mean_squared_error(test, naive_forecast))
    arima_rmse = np.sqrt(mean_squared_error(test, eval_forecast))

    # --- Production: fit on ALL data, forecast next month ---
    adf_full = adfuller(ts)[1]
    d_full = 0 if adf_full < 0.05 else 1

    prod_model = ARIMA(ts, order=(1, d_full, 1))
    prod_fit = prod_model.fit()
    next_month_forecast = prod_fit.forecast(steps=1)

    results = {
        'zip_code': zip_code,
        'ts': ts,
        'next_month_forecast': next_month_forecast,
        'naive_rmse': naive_rmse,
        'arima_rmse': arima_rmse,
        'd': d_full,
    }
    return results

## 2. Pickle the model

In [22]:
import pickle

all_results = {}
skipped = []

for zc in df['mod_zcta'].unique():
    result = fit_pipeline(df, zc)
    if result is None:
        skipped.append(zc)
        print(f"Skipped (constant or too short): {zc}")
    else:
        all_results[zc] = result
        print(f"Done: {zc}")

print(f"\nTotal modeled: {len(all_results)}, Skipped: {len(skipped)}")

# Save for Streamlit to load
with open('forecast_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)

/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Done: 11692
Done: 10075


/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Done: 11427
Done: 11224


/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


...

Done: 10457
Done: 10040


/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Done: 11420
Done: 11378
Done: 10036
Done: 11209


/opt/miniconda3/envs/data-analysis-env/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


Done: 11211
Skipped (constant or too short): 10000

Total modeled: 177, Skipped: 1
